<a href="https://colab.research.google.com/github/luciaPi/MLSS2026-generative-models/blob/main/1_VAE_MNIST_cisty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Variačný Autoencoder (VAE) na MNIST



## 1. Import a príprava dát

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Nastavenie zariadenia
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používam zariadenie: {device}")

In [ ]:
# Hyperparametre
BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3
LATENT_DIM = 20  # Veľkosť latentného priestoru

# Načítanie MNIST datasetu
transform = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST('./data', train=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Trénovacích vzoriek: {len(train_dataset)}")
print(f"Testovacích vzoriek: {len(test_dataset)}")

## 2. Definícia VAE modelu

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent_dim=20):
        super(VAE, self).__init__()

        # ENCODER
        self.fc1 = nn.Linear(784, 400)
        self.fc_mu = nn.Linear(400, latent_dim) # each dimension of latent space has its own mean and variance
        self.fc_logvar = nn.Linear(400, latent_dim) # logarithm of variance (because variance have to by positive, and we use exp in reparametrize)

        # DECODER
        self.fc3 = nn.Linear(latent_dim, 400)
        self.fc4 = nn.Linear(400, 784)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std) # random from N(0,1) with shape same as std
        z = mu + eps * std
        return z

    def decode(self, z):
        h = F.relu(self.fc3(z))
        recon = torch.sigmoid(self.fc4(h)) # sigmoid normalise to (0,1)
        return recon

    def forward(self, x):
        mu, logvar = self.encode(x.view(-1, 784))
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar

model = VAE(latent_dim=LATENT_DIM).to(device)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"\nModel architektúra:")
print(model)
print(f"\nPočet parametrov: {sum(p.numel() for p in model.parameters())}")

## 3. Loss funkcia

In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    BCE = F.binary_cross_entropy(recon_x, x.view(-1, 784), reduction='sum') # pixel values in interval (0,1)
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) # KL divergence
    return BCE + KLD, BCE, KLD

## 4. Trénovanie

In [ ]:
def train(epoch):
    model.train()
    train_loss = 0
    train_bce = 0
    train_kld = 0

    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        optimizer.zero_grad()

        recon_batch, mu, logvar = model(data)
        loss, bce, kld = vae_loss(recon_batch, data, mu, logvar)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_bce += bce.item()
        train_kld += kld.item()

        if batch_idx % 100 == 0:
            print(f'Epocha {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] '
                  f'Loss: {loss.item() / len(data):.4f}')

    avg_loss = train_loss / len(train_loader.dataset)
    avg_bce = train_bce / len(train_loader.dataset)
    avg_kld = train_kld / len(train_loader.dataset)

    print(f'====> Epocha {epoch} Priemerný loss: {avg_loss:.4f} '
          f'(BCE: {avg_bce:.4f}, KLD: {avg_kld:.4f})')

    return avg_loss, avg_bce, avg_kld

print("="*50)
print("ZAČÍNAM TRÉNOVANIE")
print("="*50)

losses = []
bce_losses = []
kld_losses = []

for epoch in range(1, EPOCHS + 1):
    loss, bce, kld = train(epoch)
    losses.append(loss)
    bce_losses.append(bce)
    kld_losses.append(kld)

## 5. Vizualizácia výsledkov

In [ ]:
# Loss krivky
plt.figure(figsize=(15, 4))

plt.subplot(1, 3, 1)
plt.plot(losses)
plt.title('Total Loss')
plt.xlabel('Epocha')
plt.ylabel('Loss')
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(bce_losses)
plt.title('Reconstruction Loss (BCE)')
plt.xlabel('Epocha')
plt.ylabel('BCE')
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(kld_losses)
plt.title('KL Divergence')
plt.xlabel('Epocha')
plt.ylabel('KLD')
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Rekonštrukcia testovacích obrázkov
model.eval()
with torch.no_grad():
    data, _ = next(iter(test_loader))
    data = data.to(device)
    recon, _, _ = model(data)

    n = 10
    plt.figure(figsize=(20, 4))
    for i in range(n):
        ax = plt.subplot(2, n, i + 1)
        plt.imshow(data[i].cpu().squeeze(), cmap='gray')
        ax.set_title('Originál')
        ax.axis('off')

        ax = plt.subplot(2, n, i + 1 + n)
        plt.imshow(recon[i].cpu().view(28, 28), cmap='gray')
        ax.set_title('Rekonštrukcia')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

## 6. Generovanie nových obrázkov

In [ ]:
print("="*50)
print("GENEROVANIE NOVÝCH OBRÁZKOV")
print("="*50)

with torch.no_grad():
    z = torch.randn(64, LATENT_DIM).to(device)
    sample = model.decode(z).cpu()

    plt.figure(figsize=(10, 10))
    for i in range(64):
        plt.subplot(8, 8, i + 1)
        plt.imshow(sample[i].view(28, 28), cmap='gray')
        plt.axis('off')

    plt.suptitle('Náhodne generované číslice', fontsize=16)
    plt.tight_layout()
    plt.show()

## 7. Interpolácia v latentnom priestore

In [ ]:
print("="*50)
print("INTERPOLÁCIA MEDZI DVOMA ČÍSLICAMI")
print("="*50)

def interpolate(model, z1, z2, steps=10):
    results = []
    for alpha in np.linspace(0, 1, steps):
        z = (1 - alpha) * z1 + alpha * z2
        with torch.no_grad():
            sample = model.decode(z).cpu()
        results.append(sample)
    return results

model.eval()
with torch.no_grad():
    data, labels = next(iter(test_loader))
    data = data.to(device)

    idx_3 = (labels == 3).nonzero()[0].item()
    idx_7 = (labels == 7).nonzero()[0].item()

    img_3 = data[idx_3:idx_3+1]
    img_7 = data[idx_7:idx_7+1]

    mu_3, logvar_3 = model.encode(img_3.view(-1, 784))
    mu_7, logvar_7 = model.encode(img_7.view(-1, 784))

    z_3 = mu_3
    z_7 = mu_7

    steps = 15
    interpolated = interpolate(model, z_3, z_7, steps)

    plt.figure(figsize=(20, 2))

    plt.subplot(1, steps + 2, 1)
    plt.imshow(img_3.cpu().squeeze(), cmap='gray')
    plt.title('Originál 3')
    plt.axis('off')

    for i, img in enumerate(interpolated):
        plt.subplot(1, steps + 2, i + 2)
        plt.imshow(img.view(28, 28), cmap='gray')
        plt.title(f'{i/(steps-1):.2f}')
        plt.axis('off')

    plt.subplot(1, steps + 2, steps + 2)
    plt.imshow(img_7.cpu().squeeze(), cmap='gray')
    plt.title('Originál 7')
    plt.axis('off')

    plt.suptitle('Interpolácia v latentnom priestore: 3 → 7', fontsize=16)
    plt.tight_layout()
    plt.show()

## 8. Vizualizácia 2D latentného priestoru

In [ ]:
print("="*50)
print("VIZUALIZÁCIA LATENTNÉHO PRIESTORU")
print("="*50)

model_2d = VAE(latent_dim=2).to(device)
optimizer_2d = optim.Adam(model_2d.parameters(), lr=LEARNING_RATE)

print("\nTrénovanie 2D VAE (3 epochy)...")
for epoch in range(1, 4):
    train_loss = 0
    for data, _ in train_loader:
        data = data.to(device)
        optimizer_2d.zero_grad()
        recon, mu, logvar = model_2d(data)
        loss, _, _ = vae_loss(recon, data, mu, logvar)
        loss.backward()
        optimizer_2d.step()
        train_loss += loss.item()
    print(f'Epocha {epoch}, Loss: {train_loss/len(train_loader.dataset):.4f}')

In [ ]:
# Vizualizácia 2D latentného priestoru
model_2d.eval()
with torch.no_grad():
    z_list = []
    label_list = []

    for data, labels in test_loader:
        data = data.to(device)
        mu, _ = model_2d.encode(data.view(-1, 784))
        z_list.append(mu.cpu())
        label_list.append(labels)

    z_array = torch.cat(z_list).numpy()
    labels_array = torch.cat(label_list).numpy()

    plt.figure(figsize=(12, 10))
    scatter = plt.scatter(z_array[:, 0], z_array[:, 1],
                         c=labels_array, cmap='tab10',
                         alpha=0.5, s=5)
    plt.colorbar(scatter, label='Číslica')
    plt.xlabel('Latentná dimenzia 1')
    plt.ylabel('Latentná dimenzia 2')
    plt.title('2D Latentný priestor VAE')
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
# Manifestácia latentného priestoru
n = 20
digit_size = 28
figure = np.zeros((digit_size * n, digit_size * n))

grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)[::-1]

model_2d.eval()
with torch.no_grad():
    for i, yi in enumerate(grid_y):
        for j, xi in enumerate(grid_x):
            z_sample = torch.tensor([[xi, yi]], dtype=torch.float).to(device)
            x_decoded = model_2d.decode(z_sample).cpu()
            digit = x_decoded.view(digit_size, digit_size).numpy()
            figure[i * digit_size: (i + 1) * digit_size,
                   j * digit_size: (j + 1) * digit_size] = digit

plt.figure(figsize=(15, 15))
plt.imshow(figure, cmap='gray')
plt.title('Manifestácia latentného priestoru')
plt.axis('off')
plt.tight_layout()
plt.show()

## Záver

**Úlohy na experimentovanie:**
1. Zmeňte LATENT_DIM (napr. 5, 50, 100) a pozorujte kvalitu
2. Upravte architektúru siete (pridajte vrstvy)
3. Experimentujte s beta-VAE: loss = BCE + beta * KLD
4. Skúste interpolovať medzi viacerými číslicami
5. Vytvorte 'aritmetiku' v latentnom priestore

Vytvorené s použitím Claude AI.